# GymLeader: SFT of Qwen3.5-2B on Google Colab

LoRA fine-tuning of Qwen3.5-2B on our workout dataset, following Unsloth's Qwen3.5 guide:
https://unsloth.ai/docs/models/qwen3.5/fine-tune

## How to run
- **Runtime:** Runtime → Change runtime type → **T4 GPU** (or better: L4, A100).
- **Run the cells in order**, one at a time, and compare each output with the section's **Check:** line.
- **After the install cell, restart the session** (Runtime → Restart session) and continue from cell 2 (the version print).
- **If Colab disconnects** during training: the runtime is new, so start again from the install (and restart), then
  re-run sections 1–6 and also 7–9, which rebuild LoRA and the trainer. The resume cell at the end of section 9 finds
  the latest checkpoint on Drive; **skip section 10** and run section 11: training continues from that checkpoint.
- `test.jsonl` is never loaded in this notebook: the test set is only for the final evaluation on the Mac.

## 1. Install

Unsloth's guide says to upgrade `unsloth` and `unsloth_zoo` with a forced, no-cache reinstall.
Qwen3.5 **needs transformers v5** (older versions can't load it), and Colab ships v4, so we ask for v5 in the same command:
pip then picks versions that work together, or fails loudly instead of silently mixing.

This takes a few minutes. Then restart the session and run from the next cell.

**Check:** pip ends without errors; after the restart, the next cell prints transformers 5.x.

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo "transformers>=5,<6"

In [ ]:
import unsloth  # import first: Unsloth patches transformers/trl when it is imported
import torch, transformers, trl

for lib in (unsloth, transformers, trl, torch):
    print(f"{lib.__name__:13} {lib.__version__}")
assert transformers.__version__.startswith("5."), "Qwen3.5 needs transformers v5: restart the session and re-run"

## Settings

Every setting of this run in one place. Change them here, nowhere else.
Effective batch size = `BATCH_SIZE × GRAD_ACCUM` = 16 rows per optimizer step, so 2,000 rows = 125 steps per epoch.

**Check:** the cell prints the run folder and 250 total steps.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-2B"
MAX_SEQ_LENGTH = 1024
LORA_R, LORA_ALPHA = 16, 16
LEARNING_RATE = 2e-4
BATCH_SIZE = 8           # rows per GPU step
GRAD_ACCUM = 2           # GPU steps per optimizer step
EPOCHS = 2
WARMUP_STEPS = 10
EVAL_STEPS, SAVE_STEPS, LOGGING_STEPS = 25, 50, 5
SEED = 42
DRIVE_DIR = "/content/drive/MyDrive/SFT-Gym-Workout-Program"
RUN_NAME = "sft_lora_r16"

import math
TOTAL_STEPS = math.ceil(2000 / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
print(f"run {RUN_NAME} -> {DRIVE_DIR}/{RUN_NAME}, {TOTAL_STEPS} total steps")

## 2. GPU check

Training speed and memory depend on the GPU. bf16 matters because we load the model in 16-bit (Unsloth advises against 4-bit for Qwen3.5);
GPUs without bf16 fall back to fp16. Qwen3.5 has custom (Mamba-style) Triton kernels that are compiled the first time they run,
which is especially slow on a T4.

**Check:** a GPU name and ~15 GB (T4) or more; bf16 True on L4/A100, False on T4 (plus the T4 note).

In [ ]:
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU:    {gpu.name}")
print(f"memory: {gpu.total_memory / 1e9:.1f} GB")
print(f"bf16:   {torch.cuda.is_bf16_supported()}")
if "T4" in gpu.name:
    print("Note: T4 detected. Qwen3.5's kernels compile slowly on T4, so the first steps will be slow. That is expected.")

## 3. Google Drive

Colab deletes its disk when the session ends. Everything we want to keep (checkpoints, logs, adapter, merged model)
goes to Drive, in one folder per run.

**Check:** Drive mounts and the run folder `.../SFT-Gym-Workout-Program/sft_lora_r16` is printed.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
OUT = Path(DRIVE_DIR)
RUN_DIR = OUT / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", RUN_DIR)

## 4. Data

We clone the repo at `main`, because we need today's code too: at the tag `data-v1`, `src/evaluate.py` doesn't exist yet,
and section 6 imports its prompt builder. The *data*, though, must be the frozen dataset we tagged as `data-v1`.
So we fetch the tags and assert that `data/processed/` on `main` is byte-for-byte the same as at `data-v1`
(`git diff --quiet` exits with 0 only when there are no differences). If someone regenerates the data, this cell stops.

Only `train.jsonl` and `val.jsonl` are loaded. **`test.jsonl` is never loaded here**: the test set is only for the final
evaluation on the Mac, and looking at it during training would make the test score meaningless.

**Check:** "data/processed/ matches data-v1" and "train 2000, val 200".

In [ ]:
import json, subprocess, sys

REPO = Path("/content/SFT-Gym-Workout-Program")
if not REPO.exists():
    !git clone --quiet --branch main https://github.com/riccardoroselli/SFT-Gym-Workout-Program.git {REPO}
!git -C {REPO} fetch --quiet --tags
!git -C {REPO} log -1 --oneline
diff = subprocess.run(["git", "-C", str(REPO), "diff", "--quiet", "data-v1", "HEAD", "--", "data/processed/"])
assert diff.returncode == 0, "data/processed/ on main differs from data-v1: the training data is not the frozen dataset"
print("data/processed/ matches data-v1")
sys.path.insert(0, str(REPO / "src"))  # so section 6 can import evaluate.py

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

train = read_jsonl(REPO / "data/processed/train.jsonl")
val = read_jsonl(REPO / "data/processed/val.jsonl")
assert (len(train), len(val)) == (2000, 200), (len(train), len(val))
print(f"train {len(train)}, val {len(val)}")
print(train[0]["messages"])

## 5. Model

Settings from Unsloth's guide: 16-bit weights (bf16 LoRA, not 4-bit QLoRA, which loses more accuracy on Qwen3.5) and
`full_finetuning=False`, because we will train a small LoRA adapter instead of all 2B weights.
`max_seq_length=1024` is plenty: our longest training row is far shorter (checked in the next section).
Unsloth's table says the 2B model needs about 5 GB for bf16 LoRA.

**Check:** the model loads and uses roughly 4–5 GB of GPU memory.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

## 6. Chat template check

The model must be trained on **exactly** the text it will see at evaluation time. `src/evaluate.py` builds the prompt
with `build_prompt()`, which we import here instead of copying, so the two can't drift apart. It ends with an empty
`<think>\n\n</think>\n\n` block (thinking off). Importing `evaluate.py` works on Colab because mlx (Mac only) is
imported only inside its `generate` step.
A training row is the whole conversation formatted the same way, so it must be that prompt, then the assistant JSON,
then the end-of-turn token `<|im_end|>`. If Unsloth's tokenizer formatted it differently, the fine-tuned model would
learn one format and be tested on another.

The reference prompt uses the plain Hugging Face tokenizer, the same one `evaluate.py` gets through mlx-lm.
Qwen3.5 is a vision-language model, so Unsloth may return a processor: we use the text tokenizer inside it.

**Check:** "template check passed", and the longest train row is about 209 tokens.

In [ ]:
from transformers import AutoTokenizer
from evaluate import build_prompt

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # processor -> its text tokenizer
reference = AutoTokenizer.from_pretrained(MODEL_NAME)

def train_text(row):
    return text_tokenizer.apply_chat_template(row["messages"], tokenize=False, enable_thinking=False)

row = train[0]
system, user, assistant = row["messages"]
text = train_text(row)
print(text)

eval_prompt = build_prompt(reference, [system, user])  # what evaluate.py sends, with no few-shot turns
END = "<|im_end|>"
assert text.startswith(eval_prompt), "training text does not start with the evaluate.py prompt"
assert text.rstrip("\n").endswith(assistant["content"] + END), "training text does not end with answer + <|im_end|>"
assert text == eval_prompt + assistant["content"] + END + "\n", "unexpected extra text"
print("template check passed: training text = evaluate.py prompt + answer JSON + <|im_end|>")

In [ ]:
lengths = [len(text_tokenizer(train_text(r))["input_ids"]) for r in train]
longest = max(range(len(train)), key=lengths.__getitem__)
print(f"longest train row: {lengths[longest]} tokens (row {longest}), max_seq_length {MAX_SEQ_LENGTH}")
assert lengths[longest] <= MAX_SEQ_LENGTH

## 7. Checks before LoRA

On a T4 the loading message may say float32, yet memory is only ~4.4 GB: 2B parameters in float32 would need ~8 GB.
So we count the parameters per dtype to see what the weights really are. The end-of-turn token must be `<|im_end|>`:
it's how the model will learn to stop after the JSON.

**Check:** almost all parameters in one 16-bit type (bfloat16, or float16 on a T4) and only a few thousand in float32;
`eos_token` is `<|im_end|>`.

In [ ]:
import collections

by_dtype = collections.Counter()
for p in model.parameters():
    by_dtype[str(p.dtype)] += p.numel()
print("model.dtype:", model.dtype)
for dtype, n in by_dtype.most_common():
    print(f"  {dtype:15} {n:>14,} parameters")
print("eos_token:", repr(text_tokenizer.eos_token), "id", text_tokenizer.eos_token_id)
assert text_tokenizer.eos_token == END

## 8. LoRA

LoRA freezes the 2B base weights and trains small extra matrices (rank `r=16`) next to the attention and MLP
projections. It needs far less memory and gives a small adapter file. Settings follow Unsloth's guide
(`alpha = r`, no dropout, no bias, Unsloth's memory-saving gradient checkpointing), with our seed 42.

We also keep a copy of the fresh LoRA weights: section 10 uses it to undo the test run.

**Check:** trainable parameters are a small share of the total (around 1% or less, not 0 and not 100%).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} of {total:,} parameters ({100 * trainable / total:.2f}%)")
FRESH_LORA = {k: v.detach().clone() for k, v in model.state_dict().items() if "lora_" in k}
print(f"saved a copy of {len(FRESH_LORA)} fresh LoRA tensors")

## 9. Datasets and trainer

Each row becomes a **prompt-completion** pair: the prompt is exactly what `evaluate.py` sends (`build_prompt`),
the completion is the answer JSON plus `<|im_end|>`. With `completion_only_loss=True` the loss is computed only on the
completion: the model learns to *write the answer*, not to repeat the request. We check this on one tokenized example:
the tokens that count in the loss must decode to exactly the JSON + `<|im_end|>`.

**Check:** both asserts pass; the printed loss tokens are the JSON answer ending in `<|im_end|>`.

In [ ]:
from datasets import Dataset

def prompt_completion(rows):
    return Dataset.from_list([{"prompt": build_prompt(text_tokenizer, r["messages"][:2]),
                               "completion": r["messages"][2]["content"] + END} for r in rows])

train_ds, val_ds = prompt_completion(train), prompt_completion(val)
assert all(p == build_prompt(reference, r["messages"][:2]) for p, r in zip(train_ds["prompt"], train))
print(train_ds, val_ds, sep="\n")
print(repr(train_ds[0]["prompt"]))
print(repr(train_ds[0]["completion"]))

The trainer settings: learning rate 2e-4 with 10 warmup steps and a cosine decay, 8-bit AdamW (less memory),
eval on val every 25 steps, a checkpoint on Drive every 50 steps (only the last 2 are kept).
bf16 is used when the GPU supports it, otherwise fp16 (T4). `make_trainer()` builds a trainer from these settings;
section 10 calls it with a few overrides for the short test run.

**Check:** the trainer builds (TRL tokenizes 2,000 + 200 rows) and prints its precision and output folder.

In [ ]:
from trl import SFTConfig, SFTTrainer

BF16 = torch.cuda.is_bf16_supported()

def make_trainer(**overrides):
    settings = dict(
        output_dir=str(CKPT_DIR), max_length=MAX_SEQ_LENGTH, completion_only_loss=True, dataset_num_proc=1,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_scheduler_type="cosine",
        optim="adamw_8bit", weight_decay=0.01, seed=SEED,
        eval_strategy="steps", eval_steps=EVAL_STEPS, logging_steps=LOGGING_STEPS,
        save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2,
        bf16=BF16, fp16=not BF16, report_to="none",
    )
    settings.update(overrides)
    return SFTTrainer(model=model, processing_class=text_tokenizer, train_dataset=train_ds,
                      eval_dataset=val_ds, args=SFTConfig(**settings))

trainer = make_trainer()
print(f"precision: {'bf16' if BF16 else 'fp16'}, checkpoints -> {CKPT_DIR}")

In [ ]:
example = trainer.train_dataset[0]               # the tokenized version of train_ds[0]
batch = trainer.data_collator([example])
ids, labels = batch["input_ids"][0], batch["labels"][0]
in_loss = ids[labels != -100]                    # -100 = ignored by the loss
print(f"{len(in_loss)} of {len(ids)} tokens count in the loss:")
print(repr(text_tokenizer.decode(in_loss)))
assert text_tokenizer.decode(ids) == train_ds[0]["prompt"] + train_ds[0]["completion"]
assert text_tokenizer.decode(in_loss) == train_ds[0]["completion"], "loss is not on exactly the answer + <|im_end|>"
print("masking check passed")

**Resume cell.** If an earlier session saved checkpoints to Drive, training continues from the latest one
(model, optimizer and schedule), instead of starting over.

**Check:** "nothing to resume, fresh start" the first time; after a disconnect, the path of the latest checkpoint
(then skip section 10).

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

RESUME = get_last_checkpoint(str(CKPT_DIR)) if CKPT_DIR.exists() else None
print("resume from:", RESUME or "nothing to resume, fresh start")

## 10. Test run (20 steps)

A short run first, to catch errors and measure speed before committing to the full run. It uses a separate trainer
that writes only to `/content/test_run` (Colab's disk), with no checkpoints and no eval, so the real checkpoint folder on
Drive stays empty. Seconds per step are measured on steps 11–20, because the first steps include the slow kernel compile.

**Check:** 4 losses (one every 5 steps), roughly going down; the estimated full-run time; peak memory well below the GPU's.

In [ ]:
import time
from transformers import TrainerCallback

class StepTimer(TrainerCallback):
    def __init__(self): self.times = []
    def on_step_end(self, args, state, control, **kwargs): self.times.append(time.time())

assert RESUME is None, "resuming: skip the test run"
timer = StepTimer()
test_trainer = make_trainer(max_steps=20, output_dir="/content/test_run", save_strategy="no", eval_strategy="no")
test_trainer.add_callback(timer)
torch.cuda.reset_peak_memory_stats()
start = time.time()
test_trainer.train()
secs = (timer.times[-1] - timer.times[9]) / (len(timer.times) - 10)
print(f"first step {timer.times[0] - start:.0f} s (includes compiling), then {secs:.2f} s/step")
print(f"peak GPU memory {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
print(f"estimated full run: {TOTAL_STEPS} steps x {secs:.2f} s = {TOTAL_STEPS * secs / 60:.0f} min, plus "
      f"{TOTAL_STEPS // EVAL_STEPS} val evals")
print("losses:", [round(e["loss"], 4) for e in test_trainer.state.log_history if "loss" in e])

**Reset.** The test run changed the LoRA weights in place, and the full run must start from fresh ones.
Re-running the LoRA cell doesn't work: the model is already wrapped, so it would stack a second adapter on top.
Rebuilding the trainer alone isn't enough either: it gives a fresh optimizer and schedule, but keeps the trained weights.
So we load back the copy of the fresh LoRA weights saved in section 8, then rebuild the trainer.

**Check:** "LoRA weights reset" and no assert error.

In [ ]:
missing, unexpected = model.load_state_dict(FRESH_LORA, strict=False)
assert not unexpected, unexpected
assert all(torch.equal(model.state_dict()[k], v) for k, v in FRESH_LORA.items())
del test_trainer
trainer = make_trainer()  # fresh optimizer and learning-rate schedule
torch.cuda.empty_cache()
print("LoRA weights reset, new trainer ready")

## 11. Full training

2 epochs = 250 optimizer steps, with a val eval every 25 steps and a checkpoint on Drive every 50.
The log history (every loss) goes to Drive as JSON, so the curves can be redrawn even after the session ends.

**Check:** training reaches step 250; total time close to the estimate from section 10; `log_history.json` saved.

In [ ]:
torch.cuda.reset_peak_memory_stats()
start = time.time()
trainer.train(resume_from_checkpoint=RESUME)
print(f"total time {(time.time() - start) / 60:.1f} min, peak GPU memory {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
(RUN_DIR / "log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
print("saved", RUN_DIR / "log_history.json")

## 12. Loss curves

Train loss is measured on the batches the model is learning from; val loss on 200 rows it never trains on.
How to read it:
- **both going down** and ending close together = the model is learning things that carry over: good;
- **val rising while train keeps falling** = overfitting: the model memorises the training rows.
  Then fewer epochs (or the checkpoint before val started rising) would be better.

**Check:** both curves go down; the final val loss is close to the final train loss.

In [ ]:
import matplotlib.pyplot as plt

history = json.loads((RUN_DIR / "log_history.json").read_text())  # from Drive: works after a restart too
train_pts = [(e["step"], e["loss"]) for e in history if "loss" in e]
val_pts = [(e["step"], e["eval_loss"]) for e in history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(7, 4))
for (pts, label, color, marker) in [(train_pts, "train loss", "#2a78d6", None), (val_pts, "val loss", "#eb6834", "o")]:
    ax.plot(*zip(*pts), label=f"{label} (final {pts[-1][1]:.4f})", color=color, linewidth=2, marker=marker, markersize=6)
ax.set(xlabel="step", ylabel="loss", title=f"{RUN_NAME}: train and val loss")
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.savefig(RUN_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"final train loss {train_pts[-1][1]:.4f} (step {train_pts[-1][0]}), "
      f"final val loss {val_pts[-1][1]:.4f} (step {val_pts[-1][0]})")

## 13. Quick check on val (not test)

A sanity check that the trained model actually writes good answers: 30 val rows, the same prompt (`build_prompt`) and
the same settings as `evaluate.py` (greedy decoding, max 512 new tokens), scored with the same `verify()`.
This is **not** the real evaluation: that runs on the Mac with MLX on the test set. The answers are saved to Drive,
so section 15 can compare against them.

**Check:** parse_ok and strict_format near 1.0, names_exist and all_correct far above the baseline (0.16 and 0.0);
truncated 0.

In [ ]:
from evaluate import FIELDS, MAX_NEW_TOKENS, mean
from verifier import verify

FastLanguageModel.for_inference(model)
IM_END_ID = text_tokenizer.convert_tokens_to_ids(END)

def answer(model, row):
    """Greedy answer to a row's request, like evaluate.py. -> (text, truncated)"""
    prompt = build_prompt(text_tokenizer, row["messages"][:2])
    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         eos_token_id=IM_END_ID, pad_token_id=text_tokenizer.pad_token_id)
    new = out[0, inputs["input_ids"].shape[1]:]
    truncated = len(new) == MAX_NEW_TOKENS and new[-1].item() != IM_END_ID
    return text_tokenizer.decode(new, skip_special_tokens=True), truncated

In [ ]:
results = []
for i, row in enumerate(val[:30], 1):
    text, truncated = answer(model, row)
    results.append({"output": text, **verify(row["meta"], text), "has_think": "<think>" in text, "truncated": truncated})
    print(f"[{i}/30] reward {results[-1]['reward']:.2f}  {text[:70]!r}")
(RUN_DIR / "val_check.json").write_text(json.dumps(results, indent=1))

print()
for k in FIELDS:
    print(f"{k:14} {mean(r[k] for r in results)}")
for row, r in list(zip(val, results))[:3]:
    print("\nUSER:  ", row["messages"][1]["content"], "\nMODEL: ", r["output"])

## 14. Save

(a) The **LoRA adapter** (a small file, plus the tokenizer): enough to rebuild the model on top of the base.
(b) The **merged model**: the LoRA weights added into the base weights, one ready-to-use 16-bit model. This is what the
Mac evaluates with MLX.

**Why we check bf16:** the Mac baseline runs the base model in bf16. On a T4 (no bf16) the merge can write float16 or
float32 weights, and the SFT result would then mix two changes (training and precision). So we read each tensor's
dtype from the saved files (only the file header, no loading) and compare it with the base checkpoint: any tensor in a
different dtype is cast back to the base's dtype, which is bf16 except a few thousand float32 parameters.

**Check:** adapter and merged files listed; every shard reports only BF16 (and a few F32); `torch_dtype` bfloat16;
"tokenizer check passed".

In [ ]:
ADAPTER_DIR, MERGED_DIR = RUN_DIR / "adapter", RUN_DIR / "merged"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
print("saved adapter and merged model")

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from safetensors.torch import load_file, save_file

TORCH_DTYPE = {"BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32}

def header_dtypes(path):
    with safe_open(str(path), "pt") as f:
        return {k: f.get_slice(k).get_dtype() for k in f.keys()}

base = header_dtypes(hf_hub_download(MODEL_NAME, "model.safetensors"))  # in the HF cache since section 5
for shard in sorted(MERGED_DIR.glob("*.safetensors")):
    saved = header_dtypes(shard)
    wrong = {k for k, d in saved.items() if d in TORCH_DTYPE and d != base.get(k, "BF16")}
    print(f"{shard.name}: {dict(collections.Counter(saved.values()))}, "
          f"{len(saved.keys() & base.keys())}/{len(saved)} names found in the base")
    if wrong:
        tensors = load_file(str(shard))
        for k in wrong:
            tensors[k] = tensors[k].to(TORCH_DTYPE[base.get(k, "BF16")])
        save_file(tensors, str(shard), metadata={"format": "pt"})
        print(f"  cast {len(wrong)} tensors back to the base dtype -> {dict(collections.Counter(header_dtypes(shard).values()))}")

config_path = MERGED_DIR / "config.json"
config = json.loads(config_path.read_text())
for c in (config, config.get("text_config", {})):
    for key in ("torch_dtype", "dtype"):
        if key in c:
            c[key] = "bfloat16"
config_path.write_text(json.dumps(config, indent=2))

In [ ]:
for folder in (ADAPTER_DIR, MERGED_DIR):
    print(folder)
    for f in sorted(folder.iterdir()):
        print(f"  {f.name:40} {f.stat().st_size / 1e6:10.1f} MB")

config = json.loads((MERGED_DIR / "config.json").read_text())
print("architectures:", config.get("architectures"))
print("torch_dtype:  ", config.get("torch_dtype", config.get("dtype")),
      "| text_config:", config.get("text_config", {}).get("torch_dtype", config.get("text_config", {}).get("dtype")))

for folder in (ADAPTER_DIR, MERGED_DIR):
    saved_tok = AutoTokenizer.from_pretrained(str(folder))
    assert all(build_prompt(saved_tok, r["messages"][:2]) == build_prompt(reference, r["messages"][:2]) for r in val), folder
print("tokenizer check passed: saved chat template gives the same build_prompt output")

## 15. Reload test

Load the merged model back from Drive, as the Mac will, and answer 3 val rows again. With greedy decoding the answers
should match section 13. We free the GPU first: the training model and the reloaded one don't both need to be in memory.

**Check:** 3 × MATCH. A DIFF whose answer still has reward 1.0 is a tiny rounding difference (merged bf16 weights vs
base + LoRA computed separately); a DIFF with a broken answer means the saved model is broken.

In [ ]:
import gc

for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

reloaded, _ = FastLanguageModel.from_pretrained(str(MERGED_DIR), max_seq_length=MAX_SEQ_LENGTH,
                                                load_in_4bit=False, load_in_16bit=True)
FastLanguageModel.for_inference(reloaded)
before = json.loads((RUN_DIR / "val_check.json").read_text())
for row, old in zip(val[:3], before[:3]):
    new, _ = answer(reloaded, row)
    print("MATCH" if new == old["output"] else "DIFF ", f"reward {verify(row['meta'], new)['reward']:.2f}", repr(new[:80]))

## 16. Next steps (on the Mac)

1. **Download `merged/`** from Drive (`MyDrive/SFT-Gym-Workout-Program/sft_lora_r16/merged/`) to
   `/Volumes/lexar/produttività/projects/SFT-Gym-Workout-Program/models/sft_lora_r16/merged/`
   (`models/` is ignored by git):
   - **Google Drive for desktop**: copy the folder from the Drive mount in Finder, or
   - **browser**: right-click `merged` → Download (Drive zips it), then unzip into `models/sft_lora_r16/`.
2. **Evaluate on the test set with MLX**, scored exactly like the baselines:
   ```bash
   conda activate GymLeaderProject
   python src/evaluate.py generate --run sft_lora_r16 --model models/sft_lora_r16/merged
   python src/evaluate.py score --run sft_lora_r16
   python src/analyze_errors.py --run sft_lora_r16
   ```
   `--model` is passed to `mlx_lm.load()`, which accepts a local folder as well as a Hugging Face name.